# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [1]:
%pip install langchain_community langchain_experimental langchain_openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [16]:
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
import faiss

In [5]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [17]:
%pip install faiss-cpu
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
docstore = InMemoryDocstore({})
vectorstore = FAISS(
    embedding_function=embeddings_model.embed_query,
    index=index,
    docstore=docstore,
    index_to_docstore_id={},
)

Note: you may need to restart the kernel to use updated packages.


C:\Users\Mercy\AppData\Local\Temp\ipykernel_45312\3045655352.py:3: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings_model = OpenAIEmbeddings()
`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [18]:
%pip install google-search-results

from langchain.agents import initialize_agent, Tool, ZeroShotAgent, AgentExecutor
from langchain.agents.agent_types import AgentType
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAI

llm = OpenAI(temperature=0)

todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)
todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
]


#prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
#suffix = """Question: {task}
#{agent_scratchpad}"""
#prompt = ZeroShotAgent.create_prompt(
    #tools,
   # prefix=prefix,
  #  suffix=suffix,
 #   input_variables=["objective", "task", "context", "agent_scratchpad"],
#)

agent_executor = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run test
agent_executor.invoke({"input": "Plan a birthday party for a 10-year-old"})

Note: you may need to restart the kernel to use updated packages.


> Entering new AgentExecutor chain...
 I should consider the child's interests and preferences
Action: TODO
Action Input: "birthday party for a 10-year-old"
Observation: 

1. Decide on a date and time for the party.
2. Choose a theme for the party (e.g. sports, princess, movie, etc.).
3. Create a guest list and send out invitations.
4. Plan the menu and purchase necessary food and drinks.
5. Decide on decorations and purchase or make them.
6. Plan and organize games and activities for the party.
7. Purchase or make party favors for guests.
8. Arrange for any necessary entertainment (e.g. magician, face painter, etc.).
9. Purchase or make a birthday cake.
10. Plan and set up a designated area for presents.
11. Make a schedule for the party, including time for food, games, and presents.
12. Prepare any necessary supplies for games and activities.
13. Set up the party space the day before or the morning of the party.
14. 

{'input': 'Plan a birthday party for a 10-year-old',
 'output': "The final answer is a detailed plan for a birthday party for a 10-year-old, including considerations for the child's interests, budget, and age-appropriate activities and entertainment."}

In [19]:
# No need to install or import again, and use the existing prompt and llm variables

llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [20]:
OBJECTIVE = "Write a weather report for SF today"

In [21]:
from langchain_experimental.autonomous_agents import BabyAGI

# Make sure you have run cell 8 to define 'vectorstore' before running this cell.

baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [22]:
baby_agi({"objective": OBJECTIVE})

C:\Users\Mercy\AppData\Local\Temp\ipykernel_45312\3867971396.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  baby_agi({"objective": OBJECTIVE})



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Thought: I should use the TODO action to create a todo list
Action: TODO
Action Input: "Create a todo list for writing a weather report for SF today"
Observation: 

1. Gather all necessary information: Start by collecting all the necessary information for writing a weather report for San Francisco today. This includes current temperature, humidity, wind speed, and any potential weather events.

2. Check reliable sources: Make sure to check reliable sources such as the National Weather Service or local news stations for accurate and up-to-date information.

3. Determine the format: Decide on the format for the weather report. Will it be a written report, a video, or a combination of both? This will help guide the rest of the process.

4. Create an outline: Before diving into writing, create an outline for the weather report. This will help organize the information 

{'objective': 'Write a weather report for SF today'}